In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam, SGD, RMSprop

In [2]:
np.random.seed(42)
n = 400

c0_f1 = np.random.normal(3, 1.5, n // 2)
c0_f2 = np.random.normal(55, 12, n // 2)
c1_f1 = np.random.normal(7, 1.8, n // 2)
c1_f2 = np.random.normal(78, 10, n // 2)

X = np.column_stack([
    np.concatenate([c0_f1, c1_f1]),
    np.concatenate([c0_f2, c1_f2])
]).astype(np.float32)
y = np.concatenate([np.zeros(n // 2), np.ones(n // 2)]).astype(np.float32)

idx = np.random.permutation(n)
X, y = X[idx], y[idx]

X_mean, X_std = X.mean(0), X.std(0)
X_norm = (X - X_mean) / X_std

split = int(0.8 * n)
Xtr, Xte = X_norm[:split], X_norm[split:]
ytr, yte = y[:split], y[split:]

print(f"Dataset: {n} samples, {X.shape[1]} features")
print(f"Train: {len(ytr)} | Test: {len(yte)}")

Dataset: 400 samples, 2 features
Train: 320 | Test: 80


# PART 1 — FIVE LEARNING RULES (NumPy only)

Each rule has a distinct update formula:
  - Hebbian:     Δw = η(x·y - λw)              unsupervised
  - Perceptron:  Δw = η(t - step(net)) · x      discrete error
  - Delta:       Δw = η(t - σ)·σ'·x             continuous gradient
  - Correlation: Δw = η(t·x - λw)               supervised Hebbian
  - Outstar:     Δwᵢ = η·xᵢ·(t - wᵢ)           direct target pull

In [3]:
class SingleNeuron:
    def __init__(self, n_feat, lr=0.01):
        self.w = np.random.randn(n_feat).astype(np.float32) * 0.1
        self.b = np.float32(0.0)
        self.lr = lr

    @staticmethod
    def sig(z):
        return 1.0 / (1.0 + np.exp(-np.clip(z, -500, 500)))

    @staticmethod
    def sig_d(z):
        s = SingleNeuron.sig(z)
        return s * (1 - s)

    def net(self, x):
        return x @ self.w + self.b

    def predict(self, X):
        return (self.sig(self.net(X)) >= 0.5).astype(int)

    def acc(self, X, y):
        return np.mean(self.predict(X) == y.astype(int))

    def acc_hebbian(self, X, y):
        """Hebbian is unsupervised — check both orientations."""
        out = self.sig(self.net(X))
        return max(
            np.mean((out >= 0.5).astype(int) == y.astype(int)),
            np.mean((out < 0.5).astype(int) == y.astype(int))
        )

    # ---- Hebbian: unsupervised, pure correlation ----
    def train_hebbian(self, X, y, epochs=200, decay=0.005):
        log = []
        for e in range(epochs):
            for xi in X:
                yo = self.sig(self.w @ xi + self.b)
                self.w = np.clip(self.w, -10, 10)
                self.b = np.clip(self.b, -10, 10)
            if (e + 1) % 50 == 0:
                log.append(self.acc_hebbian(Xtr, ytr))
        return log

    # ---- Perceptron: step function, updates only on errors ----
    def train_perceptron(self, X, y, epochs=200):
        log = []
        for e in range(epochs):
            for xi, t in zip(X, y):
                net = self.w @ xi + self.b
                pred = 1.0 if net >= 0 else 0.0
                self.w += self.lr * (t - pred) * xi
                self.b += self.lr * (t - pred)
            if (e + 1) % 50 == 0:
                log.append(self.acc(Xtr, ytr))
        return log

    # ---- Delta (Widrow-Hoff): continuous gradient ----
    def train_delta(self, X, y, epochs=200):
        log = []
        for e in range(epochs):
            for xi, t in zip(X, y):
                net = self.w @ xi + self.b
                out = self.sig(net)
                grad = self.sig_d(net)
                self.w += self.lr * (t - out) * grad * xi
                self.b += self.lr * (t - out) * grad
            if (e + 1) % 50 == 0:
                log.append(self.acc(Xtr, ytr))
        return log

    # ---- Correlation: supervised Hebbian (target replaces output) ----
    def train_correlation(self, X, y, epochs=200, decay=0.005):
        log = []
        for e in range(epochs):
            for xi, t in zip(X, y):
                self.w += self.lr * t * xi
                self.b += self.lr * t
            self.w = np.clip(self.w, -10, 10)
            self.b = np.clip(self.b, -10, 10)
            if (e + 1) % 50 == 0:
                log.append(self.acc(Xtr, ytr))
        return log

    # ---- Outstar: weights move directly toward target ----
    def train_outstar(self, X, y, epochs=200):
        log = []
        for e in range(epochs):
            for xi, t in zip(X, y):
                self.w += self.lr * xi * (t - self.w)
                self.b += self.lr * (t - self.b)
            if (e + 1) % 50 == 0:
                log.append(self.acc(Xtr, ytr))
        return log


# Train all five rules with identical initial weights
rule_defs = {
    'Hebbian':     ('train_hebbian',    True),
    'Perceptron':  ('train_perceptron', False),
    'Delta':       ('train_delta',      False),
    'Correlation': ('train_correlation', False),
    'Outstar':     ('train_outstar',    False),
}

rule_results = {}

for name, (method, is_hebbian) in rule_defs.items():
    np.random.seed(42)
    neuron = SingleNeuron(n_feat=Xtr.shape[1], lr=0.01)
    history = getattr(neuron, method)(Xtr, ytr, epochs=200)

    eval_fn = neuron.acc_hebbian if is_hebbian else neuron.acc
    rule_results[name] = {
        'train': eval_fn(Xtr, ytr),
        'test':  eval_fn(Xte, yte),
        'w':     neuron.w.copy(),
        'b':     neuron.b,
        'log':   history,
    }

# ---- Results table ----
print("\n" + "=" * 72)
print("  PART 1: LEARNING RULES COMPARISON (Single Neuron, lr=0.01, 200 epochs)")
print("=" * 72)

print(f"\n  {'Rule':<14} {'Train':>8} {'Test':>8}  {'Weights':>22} {'Bias':>8}")
print(f"  {'─'*14} {'─'*8} {'─'*8}  {'─'*22} {'─'*8}")
for name, r in rule_results.items():
    ws = f"[{r['w'][0]:+.3f}, {r['w'][1]:+.3f}]"
    print(f"  {name:<14} {r['train']:>7.1%} {r['test']:>7.1%}  {ws:>22} {r['b']:>+8.3f}")

# ---- Convergence speed ----
print(f"\n  Learning Progress (train accuracy at intervals)")
print(f"  {'Rule':<14} {'Ep 50':>8} {'Ep 100':>8} {'Ep 150':>8} {'Ep 200':>8}")
print(f"  {'─'*14} {'─'*8} {'─'*8} {'─'*8} {'─'*8}")
for name, r in rule_results.items():
    vals = r['log']
    print(f"  {name:<14} {vals[0]:>7.1%} {vals[1]:>7.1%} {vals[2]:>7.1%} {vals[3]:>7.1%}")


  PART 1: LEARNING RULES COMPARISON (Single Neuron, lr=0.01, 200 epochs)

  Rule              Train     Test                 Weights     Bias
  ────────────── ──────── ────────  ────────────────────── ────────
  Hebbian          80.6%   82.5%        [+0.050, -0.014]   +0.000
  Perceptron       92.2%   87.5%        [+0.017, +0.024]   +0.010
  Delta            95.9%   90.0%        [+2.898, +2.458]   +0.131
  Correlation      83.4%   82.5%      [+10.000, +10.000]  +10.000
  Outstar          95.6%   90.0%       [+10.453, +8.271]   +0.505

  Learning Progress (train accuracy at intervals)
  Rule              Ep 50   Ep 100   Ep 150   Ep 200
  ────────────── ──────── ──────── ──────── ────────
  Hebbian          80.6%   80.6%   80.6%   80.6%
  Perceptron       93.8%   91.6%   93.8%   92.2%
  Delta            95.6%   95.6%   95.6%   95.9%
  Correlation      83.4%   83.4%   83.4%   83.4%
  Outstar          95.6%   95.6%   95.6%   95.6%


# PART 2 — KERAS MLP AUTO-COMPARISON

Automatically tests every combination of:
  - 4 activation functions
  - 4 learning rates

Then compares 3 optimizers at the best activation.


In [4]:
activations = ['relu', 'tanh', 'sigmoid', 'elu']
learning_rates = [0.0001, 0.001, 0.01, 0.1]

# ---- Table 1: Activation × Learning Rate (Adam) ----
results_act_lr = []

for act in activations:
    for lr in learning_rates:
        tf.random.set_seed(42)
        model = Sequential([
            Dense(8, input_dim=Xtr.shape[1], activation=act),
            Dense(4, activation=act),
            Dense(1, activation='sigmoid')
        ])
        model.compile(optimizer=Adam(learning_rate=lr),
                      loss='binary_crossentropy', metrics=['accuracy'])
        h = model.fit(Xtr, ytr, epochs=50, verbose=0,
                      validation_data=(Xte, yte))

        results_act_lr.append({
            'act': act, 'lr': lr,
            'train': h.history['accuracy'][-1],
            'test':  h.history['val_accuracy'][-1],
            'loss':  h.history['loss'][-1],
        })

best_act_lr = max(results_act_lr, key=lambda r: r['test'])
best_act = best_act_lr['act']

print("\n" + "=" * 82)
print("  PART 2a: ACTIVATION × LEARNING RATE (Adam optimizer)")
print("=" * 82)
print(f"\n  {'Activation':<12} {'LR':>8} {'Train':>8} {'Test':>8} {'Loss':>8} {'Gap':>8}")
print(f"  {'─'*12} {'─'*8} {'─'*8} {'─'*8} {'─'*8} {'─'*8}")
for r in results_act_lr:
    gap = r['train'] - r['test']
    mark = " ★" if r is best_act_lr else ""
    print(f"  {r['act']:<12} {r['lr']:>8.4f} {r['train']:>7.1%} {r['test']:>7.1%} "
          f"{r['loss']:>8.4f} {gap:>7.1%}{mark}")

# ---- Table 2: Optimizer comparison (best activation) ----
optimizer_map = {
    'Adam':    lambda lr: Adam(learning_rate=lr),
    'SGD':     lambda lr: SGD(learning_rate=lr),
    'RMSprop': lambda lr: RMSprop(learning_rate=lr),
}

results_opt = []
for opt_name, opt_fn in optimizer_map.items():
    for lr in [0.001, 0.01, 0.1]:
        tf.random.set_seed(42)
        model = Sequential([
            Dense(8, input_dim=Xtr.shape[1], activation=best_act),
            Dense(4, activation=best_act),
            Dense(1, activation='sigmoid')
        ])
        model.compile(optimizer=opt_fn(lr),
                      loss='binary_crossentropy', metrics=['accuracy'])
        h = model.fit(Xtr, ytr, epochs=50, verbose=0,
                      validation_data=(Xte, yte))
        results_opt.append({
            'opt': opt_name, 'lr': lr,
            'train': h.history['accuracy'][-1],
            'test':  h.history['val_accuracy'][-1],
        })

best_opt = max(results_opt, key=lambda r: r['test'])

print(f"\n" + "=" * 72)
print(f"  PART 2b: OPTIMIZER COMPARISON (activation={best_act})")
print("=" * 72)
print(f"\n  {'Optimizer':<12} {'LR':>8} {'Train':>8} {'Test':>8} {'Gap':>8}")
print(f"  {'─'*12} {'─'*8} {'─'*8} {'─'*8} {'─'*8}")
for r in results_opt:
    gap = r['train'] - r['test']
    mark = " ★" if r is best_opt else ""
    print(f"  {r['opt']:<12} {r['lr']:>8.4f} {r['train']:>7.1%} {r['test']:>7.1%} "
          f"{gap:>7.1%}{mark}")


c:\tfenv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)



  PART 2a: ACTIVATION × LEARNING RATE (Adam optimizer)

  Activation         LR    Train     Test     Loss      Gap
  ──────────── ──────── ──────── ──────── ──────── ────────
  relu           0.0001   91.6%   88.7%   0.5878    2.8%
  relu           0.0010   95.6%   88.7%   0.1449    6.9%
  relu           0.0100   95.9%   90.0%   0.1197    5.9%
  relu           0.1000   96.2%   88.7%   0.1024    7.5%
  tanh           0.0001   78.8%   71.2%   0.5440    7.5%
  tanh           0.0010   95.6%   87.5%   0.1887    8.1%
  tanh           0.0100   95.6%   91.3%   0.1041    4.4% ★
  tanh           0.1000   95.3%   87.5%   0.0930    7.8%
  sigmoid        0.0001   49.1%   53.8%   0.7102   -4.7%
  sigmoid        0.0010   93.8%   86.3%   0.4990    7.5%
  sigmoid        0.0100   95.9%   91.3%   0.1236    4.7%
  sigmoid        0.1000   95.9%   91.3%   0.0948    4.7%
  elu            0.0001   54.4%   61.3%   0.6798   -6.9%
  elu            0.0010   95.6%   90.0%   0.1273    5.6%
  elu            0.0100

# PART 3 — BEST MODEL PREDICTIONS


In [5]:
# =============================================================
# PART 3 — BEST MODEL PREDICTIONS
# =============================================================

print(f"\n" + "=" * 72)
print(f"  PART 3: BEST MODEL PREDICTIONS")
print(f"  Config: {best_opt['opt']}, lr={best_opt['lr']}, activation={best_act}")
print("=" * 72)

tf.random.set_seed(42)
best_model = Sequential([
    Dense(8, input_dim=Xtr.shape[1], activation=best_act),
    Dense(4, activation=best_act),
    Dense(1, activation='sigmoid')
])
best_model.compile(optimizer=optimizer_map[best_opt['opt']](best_opt['lr']),
                   loss='binary_crossentropy', metrics=['accuracy'])
best_model.fit(Xtr, ytr, epochs=50, verbose=0)

# Use values within the training data range
new_data = np.array([
    [2.0,  50],    # clear class 0
    [8.0,  85],    # clear class 1
    [5.0,  65],    # middle ground
    [6.5,  75],    # leaning class 1
    [3.0,  48],    # clear class 0
    [9.0,  90],    # strong class 1
], dtype=np.float32)

new_norm = (new_data - X_mean) / X_std
preds = best_model.predict(new_norm, verbose=0).flatten()

print(f"\n  {'Feature 1':>12} {'Feature 2':>12} {'Confidence':>12} {'Class':>8}")
print(f"  {'─'*12} {'─'*12} {'─'*12} {'─'*8}")
for row, p in zip(new_data, preds):
    label = "Class 1" if p >= 0.5 else "Class 0"
    conf = p if p >= 0.5 else 1 - p
    print(f"  {row[0]:>12.1f} {row[1]:>12.1f} {conf:>11.1%} {label:>8}")


  PART 3: BEST MODEL PREDICTIONS
  Config: Adam, lr=0.01, activation=tanh

     Feature 1    Feature 2   Confidence    Class
  ──────────── ──────────── ──────────── ────────
           2.0         50.0       99.7%  Class 0
           8.0         85.0       99.1%  Class 1
           5.0         65.0       67.7%  Class 1
           6.5         75.0       98.2%  Class 1
           3.0         48.0       99.7%  Class 0
           9.0         90.0       99.2%  Class 1


# PART 4 - SUMMARY

In [6]:
best_rule = max(rule_results.items(), key=lambda r: r[1]['test'])

print(f"\n" + "=" * 72)
print(f"  SUMMARY")
print("=" * 72)
print(f"\n  Best single-neuron rule:    {best_rule[0]} "
      f"(test acc: {best_rule[1]['test']:.1%})")
print(f"  Best Keras MLP config:      {best_opt['opt']} + {best_act} "
      f"@ lr={best_opt['lr']} (test acc: {best_opt['test']:.1%})")
print(f"  Improvement from MLP:       "
      f"{best_opt['test'] - best_rule[1]['test']:+.1%}")


  SUMMARY

  Best single-neuron rule:    Delta (test acc: 90.0%)
  Best Keras MLP config:      Adam + tanh @ lr=0.01 (test acc: 91.3%)
  Improvement from MLP:       +1.3%
